In [ ]:
import pandas as pd
import numpy as np
import json
import os
import hashlib
import pickle
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Utiliser XGBoost si installé, sinon GradientBoostingClassifier (équivalent sklearn)
try:
    from xgboost import XGBClassifier
    USE_XGBOOST = True
except ImportError:
    USE_XGBOOST = False
    print("   INFO : XGBoost non installé → GradientBoostingClassifier (sklearn) utilisé.")
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Répertoires de sortie ──────────────────────────────────
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

print("=" * 65)
print("  ENTRAÎNEMENT DES 3 MODÈLES — DÉTECTION DE FRAUDE BANCAIRE")
print("=" * 65)

  ENTRAÎNEMENT DES 3 MODÈLES — DÉTECTION DE FRAUDE BANCAIRE


In [ ]:
# ─────────────────────────────────────────────────────────
# 1. CHARGEMENT ET PRÉPARATION DES DONNÉES
# ─────────────────────────────────────────────────────────
print("\n[1/6] Chargement du dataset (avec features engineerées)...")

# Utiliser le dataset avec features engineerées si disponible
DATA_PATH = ("data/transactions_engineered.csv"
             if os.path.exists("data/transactions_engineered.csv")
             else "data/transactions_bancaires.csv")
print(f"   Source : {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"   {len(df):,} transactions | {df.shape[1]} colonnes")
print(f"   Taux de fraude : {df['fraude'].mean()*100:.2f}%")

# ── Features originales ──────────────────────────────────
NUMERIC_BASE = [
    "heure", "jour_semaine", "est_weekend",
    "montant_mad", "est_etranger",
    "delta_km", "delta_min_last_tx", "nb_tx_1h",
    "est_nouveau_device",
    "age_client", "age_compte_jours",
    "ratio_montant_moy", "risque_horaire",
]

# ── Nouvelles features EDA (ajoutées si dataset engineeré) ──
ENGINEERED_FEATURES = [f for f in [
    "log_montant_mad",     # log1p(montant_mad) — asymétrie corrigée
    "log_delta_km",        # log1p(delta_km)
    "est_nuit",            # heure <= 5 — corrélation +0.75 avec fraude
    "vitesse_km_min",      # delta_km / (delta_min_last_tx+1) — ratio 3465×
    "montant_x_etranger",  # montant_mad * est_etranger — interaction forte
    "risque_x_nb_tx",      # risque_horaire * nb_tx_1h
] if f in df.columns]

NUMERIC_FEATURES = NUMERIC_BASE + ENGINEERED_FEATURES

if ENGINEERED_FEATURES:
    print(f"   Features engineerées actives : {len(ENGINEERED_FEATURES)}")
    for f in ENGINEERED_FEATURES:
        print(f"     + {f}")

# ── Features catégorielles ──────────────────────────────
CATEGORICAL_FEATURES = [
    "type_transaction", "device_type",
    "segment_revenu", "type_carte",
]

TARGET = "fraude"

# Encodage des catégorielles
le_dict = {}
df_model = df.copy()
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_model[col + "_enc"] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

ENCODED_CAT  = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENCODED_CAT

print(f"   Total features utilisées : {len(ALL_FEATURES)}")

X = df_model[ALL_FEATURES].values
y = df_model[TARGET].values

# Train / Test split stratifié (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Poids de classe pour compenser le déséquilibre (alternative à SMOTE)
sample_weights_train = compute_sample_weight("balanced", y_train)

print(f"   Train : {len(X_train):,} | Test : {len(X_test):,}")
print(f"   Fraudes dans train : {y_train.sum():,} ({y_train.mean()*100:.2f}%)")



[1/6] Chargement du dataset...
   50,000 transactions chargées
   Taux de fraude : 2.48%
   Train : 40,000 | Test : 10,000
   Fraudes dans train : 992 (2.48%)


In [ ]:
# ─────────────────────────────────────────────────────────
# 2. DÉFINITION DES MODÈLES
# ─────────────────────────────────────────────────────────
print("\n[2/6] Configuration des modèles...")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Sauvegarde du scaler
with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

models_config = {

    # ── MODÈLE 1 : Random Forest ─────────────────────────
    # Forces  : robuste au bruit, pas besoin de scaling, feature importance native
    # Faiblesses : plus lent à l'inférence que Logistic Regression
    # Utilisation recommandée : quand l'interprétabilité ET les perfs sont importantes
    "random_forest": {
        "name": "Random Forest",
        "short": "RF",
        "model": RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features="sqrt",
            class_weight="balanced",   # gère le déséquilibre automatiquement
            n_jobs=-1,
            random_state=42,
        ),
        "needs_scaling": False,
        "description": (
            "Ensemble de 300 arbres de décision. Robuste aux outliers et au bruit. "
            "class_weight='balanced' compense le déséquilibre fraude/légitime. "
            "Donne directement les feature importances pour SHAP."
        ),
    },

    # ── MODÈLE 2 : XGBoost ───────────────────────────────
    # Forces  : meilleur AUC sur données déséquilibrées, gère nativement
    #           le déséquilibre via scale_pos_weight
    # Faiblesses : hyperparamètres plus nombreux à tuner
    # Utilisation recommandée : production bancaire — MEILLEUR pour la fraude
    #
    # Paramètres XGBoost (différents de sklearn GradientBoosting !) :
    #   scale_pos_weight = n_negatifs / n_positifs = 49000/1000 = 49
    #   → compense le déséquilibre directement dans la fonction de perte
    #   colsample_bytree = fraction de features par arbre (≈ max_features sklearn)
    #   min_child_weight  = minimum de samples dans une feuille (≈ min_samples_leaf)
    #   gamma             = minimum de réduction de loss pour splitter (≈ min_split)
    "gradient_boosting": {
        "name": "XGBoost" if USE_XGBOOST else "Gradient Boosting (sklearn)",
        "short": "XGB",
        "model": (
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=5,
                scale_pos_weight=15,
                colsample_bytree=0.8,
                subsample=0.8,
                min_child_weight=10,
                gamma=1,
                reg_alpha=0.1,
                reg_lambda=1.0,
                use_label_encoder=False,
                eval_metric="aucpr",
                random_state=42,
                n_jobs=-1,
            ) if USE_XGBOOST else
            GradientBoostingClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=5,
                subsample=0.8,
                max_features="sqrt",
                min_samples_leaf=10,
                min_samples_split=20,
                random_state=42,
            )
        ),
        "needs_scaling": False,
        "description": (
            "XGBoost (scale_pos_weight=15, calibré EDA). "
            "colsample_bytree=0.8 · min_child_weight=10 · gamma=1 · eval_metric=aucpr."
            if USE_XGBOOST else
            "GradientBoostingClassifier sklearn — équivalent XGBoost. "
            "Utiliser XGBoost en production (pip install xgboost)."
        ),
    },

    # ── MODÈLE 3 : Régression Logistique ─────────────────
    # Forces  : très rapide en inférence (< 1ms), très interprétable
    # Faiblesses : capture mal les interactions non-linéaires
    # Utilisation recommandée : baseline ou quand la latence est critique (< 5ms)
    "logistic_regression": {
        "name": "Logistic Regression",
        "short": "LR",
        "model": LogisticRegression(
            C=0.1,                    # régularisation L2
            class_weight="balanced",
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
        ),
        "needs_scaling": True,       # nécessite les features normalisées
        "description": (
            "Modèle linéaire régularisé (L2, C=0.1). Très rapide en inférence "
            "— adapté à la contrainte < 100ms du pipeline temps réel. "
            "class_weight='balanced' + scaling StandardScaler obligatoire. "
            "Meilleure interprétabilité des coefficients (feature importance directe)."
        ),
    },
}



[2/6] Configuration des modèles...


In [ ]:
# ─────────────────────────────────────────────────────────
# 3. ENTRAÎNEMENT ET ÉVALUATION
# ─────────────────────────────────────────────────────────
print("\n[3/6] Entraînement des modèles...")

results = {}
THRESHOLD = 0.5   # seuil de classification (ajustable selon coût métier)

for model_key, cfg in models_config.items():
    print(f"\n   ── {cfg['name']} ──")

    X_tr = X_train_sc if cfg["needs_scaling"] else X_train
    X_te = X_test_sc  if cfg["needs_scaling"] else X_test

    # Entraînement
    t0 = datetime.now()
    # XGBoost gère le déséquilibre via scale_pos_weight.
    # GradientBoostingClassifier n'a pas class_weight → sample_weight
    if model_key == "gradient_boosting" and not USE_XGBOOST:
        cfg["model"].fit(X_tr, y_train, sample_weight=sample_weights_train)
    else:
        cfg["model"].fit(X_tr, y_train)
    train_time = (datetime.now() - t0).total_seconds()

    # Prédictions
    y_prob  = cfg["model"].predict_proba(X_te)[:, 1]
    y_pred  = (y_prob >= THRESHOLD).astype(int)

    # Métriques complètes
    auc_roc  = roc_auc_score(y_test, y_prob)
    auc_pr   = average_precision_score(y_test, y_prob)
    f1       = f1_score(y_test, y_pred)
    precision= precision_score(y_test, y_pred, zero_division=0)
    recall   = recall_score(y_test, y_pred)
    cm       = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Cross-validation AUC (5-fold)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(cfg["model"], X_tr, y_train, cv=cv,
                                 scoring="roc_auc", n_jobs=-1)

    print(f"     AUC-ROC   : {auc_roc:.4f}")
    print(f"     AUC-PR    : {auc_pr:.4f}  ← clé pour données déséquilibrées")
    print(f"     F1-Score  : {f1:.4f}")
    print(f"     Précision : {precision:.4f}")
    print(f"     Rappel    : {recall:.4f}  ← ne pas manquer de fraudes")
    print(f"     CV AUC    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"     Faux Neg. : {fn}  ← fraudes non détectées (critique!)")
    print(f"     Faux Pos. : {fp}  ← clients bloqués à tort")
    print(f"     Temps entr: {train_time:.1f}s")

    results[model_key] = {
        "name":       cfg["name"],
        "auc_roc":    round(auc_roc, 4),
        "auc_pr":     round(auc_pr, 4),
        "f1":         round(f1, 4),
        "precision":  round(precision, 4),
        "recall":     round(recall, 4),
        "cv_auc_mean":round(float(cv_scores.mean()), 4),
        "cv_auc_std": round(float(cv_scores.std()), 4),
        "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
        "train_time_s": round(train_time, 2),
        "description": cfg["description"],
    }

    # Sauvegarde du modèle
    model_path = f"models/{model_key}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(cfg["model"], f)

    # Hash SHA-256 du modèle (pour blockchain)
    model_hash = hashlib.sha256(open(model_path, "rb").read()).hexdigest()
    results[model_key]["model_hash"] = model_hash
    results[model_key]["model_path"] = model_path
    print(f"     Hash SHA-256 : {model_hash[:32]}...")



[3/6] Entraînement des modèles...



   ── Random Forest ──
     AUC-ROC   : 0.9503
     AUC-PR    : 0.8861  ← clé pour données déséquilibrées
     F1-Score  : 0.9313
     Précision : 0.9954
     Rappel    : 0.8750  ← ne pas manquer de fraudes
     CV AUC    : 0.8928 ± 0.0174
     Faux Neg. : 31  ← fraudes non détectées (critique!)
     Faux Pos. : 1  ← clients bloqués à tort
     Temps entr: 7.5s
     Hash SHA-256 : 4ee395fd183024e7b2e9016697625ef3...

   ── XGBoost ──
     AUC-ROC   : 0.9468
     AUC-PR    : 0.8864  ← clé pour données déséquilibrées
     F1-Score  : 0.9234
     Précision : 0.9775
     Rappel    : 0.8750  ← ne pas manquer de fraudes
     CV AUC    : 0.8971 ± 0.0119
     Faux Neg. : 31  ← fraudes non détectées (critique!)
     Faux Pos. : 5  ← clients bloqués à tort
     Temps entr: 1.1s
     Hash SHA-256 : 8375a6869911bb1f16aaee17dbca15b9...

   ── Logistic Regression ──
     AUC-ROC   : 0.9336
     AUC-PR    : 0.8832  ← clé pour données déséquilibrées
     F1-Score  : 0.7905
     Précision : 0.7209
  

In [ ]:
# ─────────────────────────────────────────────────────────
# 4. COMPARAISON ET SÉLECTION DU MEILLEUR MODÈLE
# ─────────────────────────────────────────────────────────
print("\n[4/6] Comparaison des modèles...")

# Le critère de sélection prioritaire pour la fraude bancaire :
# AUC-PR (Average Precision) > AUC-ROC > F1
# Pourquoi AUC-PR ? Parce que les données sont très déséquilibrées.
# AUC-ROC peut être bon même avec beaucoup de faux positifs.

best_key = max(results, key=lambda k: results[k]["auc_pr"])
best = results[best_key]

print(f"\n   ┌─────────────────────────────────────────────────────┐")
print(f"   │  MEILLEUR MODÈLE : {best['name']:<34}│")
print(f"   │  AUC-ROC : {best['auc_roc']:.4f}   AUC-PR : {best['auc_pr']:.4f}           │")
print(f"   │  F1      : {best['f1']:.4f}   Rappel : {best['recall']:.4f}           │")
print(f"   └─────────────────────────────────────────────────────┘")



[4/6] Comparaison des modèles...

   ┌─────────────────────────────────────────────────────┐
   │  MEILLEUR MODÈLE : XGBoost                           │
   │  AUC-ROC : 0.9468   AUC-PR : 0.8864           │
   │  F1      : 0.9234   Rappel : 0.8750           │
   └─────────────────────────────────────────────────────┘


In [ ]:
# ─────────────────────────────────────────────────────────
# 5. FEATURE IMPORTANCE (SHAP simplifié)
# ─────────────────────────────────────────────────────────
print("\n[5/6] Génération des feature importances...")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle("Feature Importance — 3 Modèles de Détection de Fraude",
             fontsize=14, fontweight="bold", y=1.02)

colors = ["#0F6E56", "#534AB7", "#854F0B"]

for idx, (model_key, cfg) in enumerate(models_config.items()):
    model = cfg["model"]
    ax = axes[idx]

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        # Régression Logistique : utiliser les coefficients absolus
        importances = np.abs(model.coef_[0])

    # Top 12 features
    indices = np.argsort(importances)[::-1][:12]
    top_features = [ALL_FEATURES[i] for i in indices]
    top_values   = importances[indices]

    # Noms lisibles
    readable = {
        "delta_km": "Distance (km)", "montant_mad": "Montant (MAD)",
        "ratio_montant_moy": "Ratio montant/moy", "nb_tx_1h": "Nb tx / heure",
        "heure": "Heure tx", "delta_min_last_tx": "Délai dernière tx",
        "est_etranger": "Transaction étrangère", "risque_horaire": "Risque horaire",
        "est_nouveau_device": "Nouveau device", "age_compte_jours": "Âge compte",
        "age_client": "Âge client", "est_weekend": "Week-end",
        "type_transaction_enc": "Type transaction", "device_type_enc": "Device type",
        "segment_revenu_enc": "Segment revenu", "type_carte_enc": "Type carte",
        "jour_semaine": "Jour semaine",
    }
    labels = [readable.get(f, f) for f in top_features]

    bars = ax.barh(range(len(labels)), top_values[::-1],
                   color=colors[idx], alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels[::-1], fontsize=9)
    ax.set_title(f"{cfg['name']}\nAUC-ROC: {results[model_key]['auc_roc']:.4f} | "
                 f"F1: {results[model_key]['f1']:.4f}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Importance", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="both", labelsize=8)

plt.tight_layout()
plt.savefig("reports/figures/feature_importance.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.close()
print("   → reports/figures/feature_importance.png")

# ── Matrice de confusion pour le meilleur modèle ─────────
best_model = models_config[best_key]["model"]
X_te_best  = X_test_sc if models_config[best_key]["needs_scaling"] else X_test
y_prob_best = best_model.predict_proba(X_te_best)[:, 1]
y_pred_best = (y_prob_best >= THRESHOLD).astype(int)
cm_best = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_best, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Prédit Légitime", "Prédit Fraude"],
            yticklabels=["Réel Légitime", "Réelle Fraude"],
            ax=ax, cbar=False, linewidths=0.5)
ax.set_title(f"Matrice de Confusion — {best['name']}\n"
             f"(seuil={THRESHOLD})", fontsize=11, fontweight="bold")
ax.set_ylabel("Réalité", fontsize=10)
ax.set_xlabel("Prédiction", fontsize=10)

# Annotations métier
tn_v, fp_v, fn_v, tp_v = cm_best.ravel()
ax.text(0.5, -0.18,
        f"Faux Négatifs (fraudes manquées) = {fn_v} | "
        f"Faux Positifs (clients bloqués) = {fp_v}",
        transform=ax.transAxes, ha="center", fontsize=8,
        color="gray", style="italic")

plt.tight_layout()
plt.savefig("reports/figures/confusion_matrix.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.close()
print("   → reports/figures/confusion_matrix.png")



[5/6] Génération des feature importances...
   → reports/figures/feature_importance.png
   → reports/figures/confusion_matrix.png


In [ ]:
# ─────────────────────────────────────────────────────────
# 6. RAPPORT JSON (format blockchain / MLflow)
# ─────────────────────────────────────────────────────────
print("\n[6/6] Génération du rapport de comparaison...")

report = {
    "experiment_name":  "fraud_detection_comparison",
    "run_date":         datetime.now().isoformat(),
    "dataset": {
        "path":              DATA_PATH,
        "sha256":            hashlib.sha256(open(DATA_PATH,"rb").read()).hexdigest(),
        "n_train":           len(X_train),
        "n_test":            len(X_test),
        "features":          ALL_FEATURES,
        "n_features":        len(ALL_FEATURES),
        "engineered_features": ENGINEERED_FEATURES,
    },
    "models":           results,
    "best_model": {
        "key":          best_key,
        "name":         best["name"],
        "selection_criterion": "AUC-PR (Average Precision Score)",
        "rationale": (
            "AUC-PR est la métrique prioritaire pour les données déséquilibrées "
            "(2% fraude). Elle mesure la qualité des prédictions sur la classe "
            "minoritaire (fraude) indépendamment des vrais négatifs (légitimes). "
            "Un modèle peut avoir AUC-ROC=0.99 et AUC-PR=0.50 si les faux positifs "
            "sont nombreux — ce qui est inacceptable en contexte bancaire."
        ),
    },
    "deployment_recommendation": {
        "production":     "gradient_boosting",
        "fallback":       "random_forest",
        "interpretable":  "logistic_regression",
        "note": (
            "En production : Gradient Boosting pour les décisions automatiques. "
            "Logistic Regression comme modèle de secours si latence critique (<5ms). "
            "En production : XGBoost avec features engineerées (EDA v2.0). "
        ),
    },
}

with open("reports/models_comparison.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# ── Résumé final ──────────────────────────────────────────
print("\n" + "="*65)
print("  RÉSUMÉ COMPARATIF")
print("="*65)
header = f"  {'Modèle':<28} {'AUC-ROC':>8} {'AUC-PR':>8} {'F1':>7} {'Rappel':>8}"
print(header)
print("  " + "-"*62)
for k, r in results.items():
    marker = " ★" if k == best_key else ""
    print(f"  {r['name']:<28} {r['auc_roc']:>8.4f} {r['auc_pr']:>8.4f} "
          f"{r['f1']:>7.4f} {r['recall']:>8.4f}{marker}")

print("\n  ★ = Meilleur modèle selon AUC-PR")
print("\n  Fichiers générés :")
print("    models/random_forest.pkl")
print("    models/gradient_boosting.pkl")
print("    models/logistic_regression.pkl")
print("    models/scaler.pkl")
print("    reports/models_comparison.json")
print("    reports/figures/feature_importance.png")
print("    reports/figures/confusion_matrix.png")
print("\n  PROCHAINE ÉTAPE : lancer 03_shap_explainability.py")
print("="*65)


[6/6] Génération du rapport de comparaison...

  RÉSUMÉ COMPARATIF
  Modèle                        AUC-ROC   AUC-PR      F1   Rappel
  --------------------------------------------------------------
  Random Forest                  0.9503   0.8861  0.9313   0.8750
  XGBoost                        0.9468   0.8864  0.9234   0.8750 ★
  Logistic Regression            0.9336   0.8832  0.7905   0.8750

  ★ = Meilleur modèle selon AUC-PR

  Fichiers générés :
    models/random_forest.pkl
    models/gradient_boosting.pkl
    models/logistic_regression.pkl
    models/scaler.pkl
    reports/models_comparison.json
    reports/figures/feature_importance.png
    reports/figures/confusion_matrix.png

  PROCHAINE ÉTAPE : lancer 03_shap_explainability.py
